In [3]:
import math
import os
import hydra
import tonic
from tonic.datasets import CIFAR10DVS
import torch
from tonic.transforms import Optional, ToFrame
from datasets.utils.pad_tensors import PadTensors
from datasets.utils.diskcache import DiskCachedDataset
import pytorch_lightning as pl
from sklearn.model_selection import train_test_split

/home/michael/projects/SE-adlif/.venv/lib/python3.11/site-packages/lightning_fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


## Data Preparation
We first need to preprocess the raw data found in the data path into a **Tonic** dataset. 

In [7]:
data_path = "/home/michael/projects/welding-data/raw"
# identify files, ending either in .raw or .bias
raw_files = []
bias_files = []
# check path
print(f"Checking path: {data_path}")
if not os.path.exists(data_path):
    raise FileNotFoundError(f"Path {data_path} does not exist.")
for file in os.listdir(data_path):
    if file.endswith(".raw"):
        raw_files.append(os.path.join(data_path, file))
    elif file.endswith(".bias"):
        bias_files.append(os.path.join(data_path, file))
print(f"Found {len(raw_files)} raw files and {len(bias_files)} bias files.")

raw_files, bias_files

Checking path: /home/michael/projects/welding-data/raw
Found 30 raw files and 30 bias files.


(['/home/michael/projects/welding-data/raw/recording12.raw',
  '/home/michael/projects/welding-data/raw/recording6.raw',
  '/home/michael/projects/welding-data/raw/recording17.raw',
  '/home/michael/projects/welding-data/raw/recording20-moving6.raw',
  '/home/michael/projects/welding-data/raw/recording9.raw',
  '/home/michael/projects/welding-data/raw/recording8.raw',
  '/home/michael/projects/welding-data/raw/recording20-welding-120cmpm.raw',
  '/home/michael/projects/welding-data/raw/recording20-welding-30cmpm.raw',
  '/home/michael/projects/welding-data/raw/recording20-60cmpm.raw',
  '/home/michael/projects/welding-data/raw/recording14.raw',
  '/home/michael/projects/welding-data/raw/recording20-moving2.raw',
  '/home/michael/projects/welding-data/raw/recording20-30cmpm.raw',
  '/home/michael/projects/welding-data/raw/recording20-welding-60cmpm.raw',
  '/home/michael/projects/welding-data/raw/recording4.raw',
  '/home/michael/projects/welding-data/raw/recording10.raw',
  '/home/mich

In [ ]:
class WeldingSpeed(tonic.datasets.Dataset):
    def __init__(self, root=data_path, transform=None, target_transform=None):
        super().__init__(root, transform, target_transform)

        self.classes = ["speed1", "speed2", "speed3"]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        events = sample["events"]
        label = sample["label"]
        if self.transform:
            events = self.transform(events)
        if self.target_transform:
            label = self.target_transform(label)
        return events, label